# Market-basket analysis on the IMDB Top 1000

## 1. Dataset and download
In this project we look for actors who often appear together in the same movie.

In [1]:
import os
os.environ['KAGGLE_USERNAME'] = "XXX"
os.environ['KAGGLE_KEY'] = "XXX"

!kaggle datasets download -d harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows
!unzip -o imdb-dataset-of-top-1000-movies-and-tv-shows.zip

Dataset URL: https://www.kaggle.com/datasets/harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows
License(s): CC0-1.0
100% 175k/175k [00:00<00:00, 357kB/s]

Archive:  imdb-dataset-of-top-1000-movies-and-tv-shows.zip
  inflating: imdb_top_1000.csv       


## 2. Loading the data and building the baskets
Here each **movie is a basket** and each **actor is an item**.
For every row we collect the names from `Star1` to `Star4`, skip empty values,
and sort them. Sorting matters, because a pair like (A, B) must always have the
same order.

instead of the full data. `random.seed(42)` makes the sample the same on every run.

In [2]:
import random
import time
import zlib
from itertools import combinations
import pandas as pd

random.seed(42)



df = pd.read_csv("imdb_top_1000.csv")
print("rows:", len(df))



rows: 1000


In [3]:
baskets = []
for i in range(len(df)):
    actors = set()
    for col in ["Star1", "Star2", "Star3", "Star4"]:
        name = df[col][i]
        if pd.notna(name):
            actors.add(name.strip())
    baskets.append(sorted(actors))

print("baskets:", len(baskets))
print("example:", baskets[0])

baskets: 1000
example: ['Bob Gunton', 'Morgan Freeman', 'Tim Robbins', 'William Sadler']


## 3. A-Priori
A-Priori finds the groups of actors that appear together in at least `s` movies
(`s` is the support threshold). It works level by level:

1. count the single actors and keep the frequent ones
2. count only the pairs whose two actors are both frequent
3. count only the triples whose pairs are all frequent, and so on

if an actor appears in fewer than `s` movies, any
pair that contains him appears in fewer than `s` movies too, so we never count it.
This keeps the number of counters small.

- `count_items`: counts single actors
- `count_level`: counts groups of a given size, skipping those with a non-frequent subset
- `keep_frequent`: keeps the itemsets with count >= `s`
- `apriori`: repeats the levels until nothing frequent is left

In [4]:
def count_items(baskets):
    counts = {}
    for basket in baskets:
        for item in basket:
            counts[(item,)] = counts.get((item,), 0) + 1
    return counts


def count_level(baskets, size, previous):
    # count only the itemsets whose smaller subsets are all frequent
    counts = {}
    for basket in baskets:
        for combo in combinations(basket, size):
            ok = True
            for sub in combinations(combo, size - 1):
                if sub not in previous:
                    ok = False
                    break
            if ok:
                counts[combo] = counts.get(combo, 0) + 1
    return counts


def keep_frequent(counts, s):
    result = {}
    for itemset in counts:
        if counts[itemset] >= s:
            result[itemset] = counts[itemset]
    return result


def apriori(baskets, s):
    frequent = {}
    current = keep_frequent(count_items(baskets), s)
    size = 2
    while len(current) > 0:
        frequent.update(current)
        counts = count_level(baskets, size, current)
        current = keep_frequent(counts, s)
        size = size + 1
    return frequent

## 4. PCY
PCY is A-Priori with an extra trick for the pairs. In the first pass, besides
counting actors, we hash every pair into one of `N_BUCKETS` counters.
Between the passes, a bucket is marked frequent if its counter is at least `s`.

In the second pass a pair is counted only if
- both actors are frequent, and
- its bucket is frequent.

`bucket_of` joins the two names with `|`, hashes the text with `crc32` (the same
result on every run, unlike Python's `hash()`) and takes `% N_BUCKETS` to get a
position from 0 to `N_BUCKETS - 1`. Two pairs can land in the same bucket
(a collision). That can make us count a pair we did not need, but it never makes
us lose a frequent one.

With 1000 movies there are at most about 6000 pairs, so 10000 buckets keeps
collisions rare.

In [5]:
N_BUCKETS = 10000

def bucket_of(pair):
    text = pair[0] + "|" + pair[1]
    return zlib.crc32(text.encode()) % N_BUCKETS


def pcy(baskets, s):
    # pass 1: count items and hash every pair into a bucket
    item_counts = {}
    buckets = [0] * N_BUCKETS
    for basket in baskets:
        for item in basket:
            item_counts[(item,)] = item_counts.get((item,), 0) + 1
        for pair in combinations(basket, 2):
            buckets[bucket_of(pair)] += 1

    frequent_items = keep_frequent(item_counts, s)
    bitmap = [count >= s for count in buckets]

    # pass 2: count a pair only if both items are frequent and its bucket is frequent
    pair_counts = {}
    for basket in baskets:
        for pair in combinations(basket, 2):
            if (pair[0],) in frequent_items and (pair[1],) in frequent_items and bitmap[bucket_of(pair)]:
                pair_counts[pair] = pair_counts.get(pair, 0) + 1
    frequent_pairs = keep_frequent(pair_counts, s)

    # triples, quadruples ... like A-Priori
    frequent = {}
    frequent.update(frequent_items)
    current = frequent_pairs
    size = 3
    while len(current) > 0:
        frequent.update(current)
        counts = count_level(baskets, size, current)
        current = keep_frequent(counts, s)
        size = size + 1
    return frequent, len(pair_counts)

## 5. SON
SON splits the baskets into chunks and uses two passes.

- **Pass 1:** run A-Priori inside each chunk with a smaller threshold,
  `ceil(s * chunk size / n)`. Every itemset that is frequent in at least one
  chunk becomes a candidate.
- **Pass 2:** count all the candidates on the whole data and keep those with
  count >= `s`.

An itemset that is frequent in the whole data must be frequent in at least one
chunk, so no frequent itemset is missed. The result is exact. The chunks are
independent.

In [6]:
def son(baskets, s, n_chunks=5):
    n = len(baskets)
    chunk_size = (n + n_chunks - 1) // n_chunks

    # pass 1: find the frequent itemsets inside every chunk (lower threshold)
    candidates = set()
    for start in range(0, n, chunk_size):
        chunk = baskets[start:start + chunk_size]
        local_s = (s * len(chunk) + n - 1) // n      # s * (chunk size / n), rounded up
        local_s = max(1, local_s)
        local = apriori(chunk, local_s)
        candidates.update(local.keys())

    # pass 2: count all the candidates on the whole data
    counts = {}
    for basket in baskets:
        for size in range(1, len(basket) + 1):
            for combo in combinations(basket, size):
                if combo in candidates:
                    counts[combo] = counts.get(combo, 0) + 1
    return keep_frequent(counts, s), len(candidates)

## 6. Checking the algorithms
To check that the three algorithms are correct, `brute_force` counts every
possible subset of every basket and keeps the ones with count >= `s`. It is slow
on big data, but fine for 1000 baskets.

We run all four methods with `s = 3` and compare. If `all equal` is `True`, then
A-Priori, PCY and SON give exactly the brute-force answer.

In [7]:
def brute_force(baskets, s):
    counts = {}
    for basket in baskets:
        for size in range(1, len(basket) + 1):
            for combo in combinations(basket, size):
                counts[combo] = counts.get(combo, 0) + 1
    return keep_frequent(counts, s)


s = 3
result_apriori = apriori(baskets, s)
result_pcy, pairs_counted = pcy(baskets, s)
result_son, n_candidates = son(baskets, s)
result_brute = brute_force(baskets, s)

print("A-Priori:", len(result_apriori), "frequent itemsets")
print("PCY:     ", len(result_pcy) )
print("SON:     ", len(result_son))
print("Brute:   ", len(result_brute))
print("all equal:", result_apriori == result_brute and result_pcy == result_brute and result_son == result_brute)

#how many distinct pairs A-Priori counts: pairs of two frequent items that occur in some movie
frequent_items = {k for k in result_apriori if len(k) == 1}
pairs_apriori = set()
for basket in baskets:
    for pair in combinations(basket, 2):
        if (pair[0],) in frequent_items and (pair[1],) in frequent_items:
            pairs_apriori.add(pair)

print()
print("pairs counted by A-Priori:", len(pairs_apriori))
print("pairs counted by PCY:     ", pairs_counted)
print("SON candidates:           ", n_candidates)

A-Priori: 299 frequent itemsets
PCY:      299
SON:      299
Brute:    299
all equal: True

pairs counted by A-Priori: 622
pairs counted by PCY:      104
SON candidates:            13492


## 7. Results for different thresholds
We run A-Priori with `s` = 2, 3, 4, 5 and 8 and count how many frequent itemsets
of each size we get (1 = single actors, 2 = pairs, 3 = triples, 4 = groups of four).
A lower `s` gives more results, and a higher `s` keeps only the strongest groups.

In [8]:
for s in [2, 3, 4, 5, 8]:
    found = apriori(baskets, s)
    by_size = {1: 0, 2: 0, 3: 0, 4: 0}
    for itemset in found:
        by_size[len(itemset)] += 1
    print("s =", s, by_size)

s = 2 {1: 641, 2: 121, 3: 25, 4: 4}
s = 3 {1: 271, 2: 25, 3: 3, 4: 0}
s = 4 {1: 145, 2: 5, 3: 1, 4: 0}
s = 5 {1: 79, 2: 3, 3: 1, 4: 0}
s = 8 {1: 17, 2: 0, 3: 0, 4: 0}


## 8. Association rules
From the frequent itemsets we build rules such as *A, B -> C*
("when A and B are in a movie, C is often there too").

- **Confidence** = support of the full itemset / support of the left side.
  It says how often the rule holds.
- **Interest** = confidence - how often C appears overall.
  A high value means C appears with A and B more than it would by chance.

We print the 10 rules with the highest confidence.

In [9]:
s = 3
frequent = apriori(baskets, s)
N = len(baskets)

rules = []
for itemset in frequent:
    if len(itemset) < 2:
        continue
    for j in itemset:
        left = tuple(x for x in itemset if x != j)
        confidence = frequent[itemset] / frequent[left]
        interest = confidence - frequent[(j,)] / N
        rules.append((confidence, interest, frequent[itemset], left, j))

rules.sort(reverse=True)
for confidence, interest, support, left, j in rules[:10]:
    print(left, "->", j, "| support", support, "| confidence", round(confidence, 2), "| interest", round(interest, 3))

('Ian McKellen', 'Orlando Bloom') -> Elijah Wood | support 3 | confidence 1.0 | interest 0.997
('Carrie Fisher', 'Harrison Ford') -> Mark Hamill | support 3 | confidence 1.0 | interest 0.997
('Mark Hamill',) -> Carrie Fisher | support 3 | confidence 1.0 | interest 0.996
('Harrison Ford', 'Mark Hamill') -> Carrie Fisher | support 3 | confidence 1.0 | interest 0.996
('Elijah Wood', 'Ian McKellen') -> Orlando Bloom | support 3 | confidence 1.0 | interest 0.996
('Elijah Wood',) -> Orlando Bloom | support 3 | confidence 1.0 | interest 0.996
('Rupert Grint',) -> Daniel Radcliffe | support 6 | confidence 1.0 | interest 0.994
('Daniel Radcliffe',) -> Rupert Grint | support 6 | confidence 1.0 | interest 0.994
('Emma Watson', 'Rupert Grint') -> Daniel Radcliffe | support 5 | confidence 1.0 | interest 0.994
('Daniel Radcliffe', 'Emma Watson') -> Rupert Grint | support 5 | confidence 1.0 | interest 0.994


# Scalability test on a larger real dataset

The IMDB Top 1000 file has only 1000 movies, which is too small to see how the
algorithms behave when the data grows. For the scalability test we use a second
Kaggle dataset, *25k Movie Dataset*, which has about 25,000 movies. The
main results above still come from the Top 1000 data.



In [10]:
!kaggle datasets download -d utsh0dey/25k-movie-dataset
!unzip -o 25k-movie-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/utsh0dey/25k-movie-dataset
License(s): CC-BY-NC-SA-4.0
100% 4.67M/4.67M [00:01<00:00, 3.59MB/s]

Archive:  25k-movie-dataset.zip
  inflating: 25k IMDb movie Dataset.csv  


In [11]:
import ast

df25 = pd.read_csv("25k IMDb movie Dataset.csv")
print(df25.columns)

baskets25 = []
casts = df25["Top 5 Casts"].dropna()

baskets25 = []
for text in casts:
    actors = ast.literal_eval(text)
    baskets25.append(sorted(set(actors)))

print("rows:", len(df25), "| baskets:", len(baskets25))
print("biggest basket:", max(len(b) for b in baskets25))
print("example:", baskets25[0])

Index(['movie title', 'Run Time', 'Rating', 'User Rating', 'Generes',
       'Overview', 'Plot Kyeword', 'Director', 'Top 5 Casts', 'Writer', 'year',
       'path'],
      dtype='object')
rows: 24402 | baskets: 24402
biggest basket: 5
example: ['Jack Epps Jr.', 'Jennifer Connelly', 'Miles Teller', 'Peter Craig', 'Tom Cruise']


## Scalability on real data
We take random samples of 1,000, 5,000, 10,000 and then all of the baskets
(24,402), and run A-Priori, PCY and SON on each one. For every size we print:

- the threshold `s` used (0.05% of the baskets, and at least 3),
- the number of frequent itemsets,
- whether the three algorithms give the same result,
- the running time of each algorithm in seconds.

The time shows how the cost grows with the amount of data, which is the main
measure of scalability here.

In [12]:
sizes = [1000, 5000, 10000, 25000]
times_apriori = []
times_pcy = []
times_son = []

for n in sizes:
    n = min(n, len(baskets25))
    part = random.sample(baskets25, n)
    s = max(3, int(0.0005 * n))

    start = time.time()
    a = apriori(part, s)
    times_apriori.append(time.time() - start)

    start = time.time()
    p, _ = pcy(part, s)
    times_pcy.append(time.time() - start)

    start = time.time()
    o, _ = son(part, s)
    times_son.append(time.time() - start)

    print(n, "baskets, s =", s, "| itemsets:", len(a), "| same result:", a == p == o,
          "| A-Priori %.1fs, PCY %.1fs, SON %.1fs" % (times_apriori[-1], times_pcy[-1], times_son[-1]))

1000 baskets, s = 3 | itemsets: 78 | same result: True | A-Priori 0.0s, PCY 0.0s, SON 0.1s
5000 baskets, s = 3 | itemsets: 1466 | same result: True | A-Priori 0.1s, PCY 0.1s, SON 0.4s
10000 baskets, s = 5 | itemsets: 1436 | same result: True | A-Priori 0.1s, PCY 0.2s, SON 0.7s
24402 baskets, s = 12 | itemsets: 1232 | same result: True | A-Priori 0.3s, PCY 0.5s, SON 0.8s


In [19]:
for s in [2, 3, 4, 5, 8]:
    result_apriori = apriori(baskets, s)
    result_pcy, _ = pcy(baskets, s)
    result_son, _ = son(baskets, s)
    result_brute = brute_force(baskets, s)

    print("s =", s,
          "| A-Priori", len(result_apriori),
          "| PCY", len(result_pcy),
          "| SON", len(result_son),
          "| Brute", len(result_brute),
          "| all equal:", result_apriori == result_brute and result_pcy == result_brute and result_son == result_brute)

s = 2 | A-Priori 791 | PCY 791 | SON 791 | Brute 791 | all equal: True
s = 3 | A-Priori 299 | PCY 299 | SON 299 | Brute 299 | all equal: True
s = 4 | A-Priori 151 | PCY 151 | SON 151 | Brute 151 | all equal: True
s = 5 | A-Priori 83 | PCY 83 | SON 83 | Brute 83 | all equal: True
s = 8 | A-Priori 17 | PCY 17 | SON 17 | Brute 17 | all equal: True


In [14]:
!pip install -q pyspark

In [15]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("market_basket").getOrCreate()
sc = spark.sparkContext

### Computations using Spark :Counting with MapReduce style
`parallelize` splits the baskets into 5 partitions. `flatMap` emits a pair
`(itemset, 1)` for every item or pair in every basket, `reduceByKey` adds the
ones for each key, and `filter` keeps the counts >= `s`. This is the same pattern as word count.



In [16]:
s = 3
baskets_rdd = sc.parallelize(baskets, 5).cache()

item_counts = (baskets_rdd
               .flatMap(lambda basket: [((item,), 1) for item in basket])
               .reduceByKey(lambda a, b: a + b)
               .filter(lambda pair: pair[1] >= s)
               .collectAsMap())

pair_counts = (baskets_rdd
               .flatMap(lambda basket: [(pair, 1) for pair in combinations(basket, 2)])
               .reduceByKey(lambda a, b: a + b)
               .filter(lambda pair: pair[1] >= s)
               .collectAsMap())

print("frequent items:", len(item_counts))
print("frequent pairs:", len(pair_counts))

# the same numbers must come out of the plain Python A-Priori
plain = apriori(baskets, s)
plain_items = {k: v for k, v in plain.items() if len(k) == 1}
plain_pairs = {k: v for k, v in plain.items() if len(k) == 2}
print("same items:", item_counts == plain_items)
print("same pairs:", pair_counts == plain_pairs)

frequent items: 271
frequent pairs: 25
same items: True
same pairs: True


### SON with Spark
Each Spark partition plays the role of a chunk.

- **Pass 1:** `mapPartitions` runs A-Priori on each partition with the lower
  threshold, and `distinct` merges the candidates.
- **Pass 2:** the candidate set is shared with `broadcast`, every basket emits
  the candidates it contains, and `reduceByKey` counts them.

The result is compared with the brute-force count.

In [17]:
def spark_son(baskets, s, n_chunks=5):
    n = len(baskets)
    rdd = sc.parallelize(baskets, n_chunks).cache()

    # pass 1: every partition (chunk) finds its own frequent itemsets
    def mine_chunk(rows):
        chunk = list(rows)
        local_s = max(1, (s * len(chunk) + n - 1) // n)
        local = apriori(chunk, local_s)
        return list(local.keys())

    candidates = rdd.mapPartitions(mine_chunk).distinct().collect()
    candidates_shared = sc.broadcast(set(candidates))

    # pass 2: count the candidates on all the baskets
    def candidates_in(basket):
        found = []
        for size in range(1, len(basket) + 1):
            for combo in combinations(basket, size):
                if combo in candidates_shared.value:
                    found.append((combo, 1))
        return found

    counts = (rdd
              .flatMap(candidates_in)
              .reduceByKey(lambda a, b: a + b)
              .filter(lambda pair: pair[1] >= s)
              .collectAsMap())
    return counts, len(candidates)


s = 3
spark_result, n_candidates = spark_son(baskets, s)
print("Spark SON:", len(spark_result), "frequent itemsets from", n_candidates, "candidates")
print("same as brute force:", spark_result == brute_force(baskets, s))

Spark SON: 299 frequent itemsets from 13492 candidates
same as brute force: True


In [20]:
random.seed(42)

# warm-up run, because the first Spark job also pays for starting up
spark_son(random.sample(baskets25, 500), 3)

sizes = [1000, 5000, 10000, 25000]
s_values = [3, 3, 5, 12]          # the thresholds from the earlier table, fixed

print("baskets |  s | plain SON | Spark SON | same result")
for i in range(len(sizes)):
    n = min(sizes[i], len(baskets25))
    s = s_values[i]
    part = random.sample(baskets25, n)

    start = time.time()
    plain_result, _ = son(part, s)
    t_plain = time.time() - start

    start = time.time()
    spark_result, _ = spark_son(part, s)
    t_spark = time.time() - start

    print("%7d | %2d | %8.2fs | %8.2fs | %s" % (n, s, t_plain, t_spark, plain_result == spark_result))

baskets |  s | plain SON | Spark SON | same result
   1000 |  3 |     0.11s |     2.03s | True
   5000 |  3 |     0.50s |     6.74s | True
  10000 |  5 |     0.94s |     2.96s | True
  24402 | 12 |     0.50s |     1.72s | True
